# Module 02 — What Is Actually in a Snapshot?

Before learning how to create and restore snapshots, it's critical to understand exactly **what data
Elasticsearch saves** — and what it does NOT save.

A snapshot can contain up to four categories of data:

```
Snapshot
├── Index data          (documents, shard files)
├── Index metadata      (mappings, settings, aliases)
├── Cluster state       (enabled with include_global_state: true)
│   ├── Persistent cluster settings
│   ├── Index templates (legacy + composable)
│   ├── Ingest pipelines
│   ├── ILM policies
│   └── Stored scripts
└── Feature states      (opt-in, per-feature)
    ├── kibana          (all Kibana saved objects)
    ├── security        (users, roles, role mappings, API keys)
    ├── watcher         (watches, watch history)
    ├── machine_learning (jobs, models)
    ├── fleet           (agent policies)
    └── ... (more)
```

In [1]:
import json
from helpers import *

es = get_client()
wait_for_green(es)

✓ Cluster health: yellow

---
## 1. Shard-Level Data

The bulk of a snapshot is Lucene segment files for each shard.
Elasticsearch deduplicates at the segment level — only **new or changed** segments are
written on subsequent snapshots (incremental by default).

In [2]:
heading('1. Shard stats for sample indices')

stats = es.indices.stats(index='kibana_sample_data_*', metric='store,docs')
t = Table('Index', 'Docs', 'Size on disk')
for idx, data in stats['indices'].items():
    t.add_row(
        idx,
        str(data['total']['docs']['count']),
        str(data['total']['store']['size_in_bytes']) + ' bytes',
    )
console.print(t)

──────────────────────────────────────── 1. Shard stats for sample indices ────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Index                                         ┃ Docs  ┃ Size on disk  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━┩
│ kibana_sample_data_ecommerce                  │ 4675  │ 4279224 bytes │
│ .ds-kibana_sample_data_logs-2026.04.06-000001 │ 14074 │ 8598979 bytes │
│ kibana_sample_data_flights                    │ 13014 │ 5976699 bytes │
└───────────────────────────────────────────────┴───────┴───────────────┘

In [3]:
heading('Lucene segment info')
segs = es.indices.segments(index='kibana_sample_data_ecommerce')
for idx, idata in segs['indices'].items():
    for shard_id, shard_data in idata['shards'].items():
        for replica in shard_data:
            for seg_name, seg in replica['segments'].items():
                console.print(f"  shard {shard_id} | {seg_name} | docs={seg['num_docs']} | size={seg['size_in_bytes']}b")

─────────────────────────────────────────────── Lucene segment info ───────────────────────────────────────────────

shard 0 | _0 | docs=2478 | size=2260137b

shard 0 | _1 | docs=2197 | size=2018662b

---
## 2. Index Metadata

Every snapshot also stores per-index metadata — mappings, settings, and aliases.
This is how ES can recreate the index structure on restore even if the target cluster has no
prior knowledge of the index.

In [4]:
heading('2. Index mappings (stored per-snapshot)')
mapping = es.indices.get_mapping(index='kibana_sample_data_ecommerce')
# Show the top-level field names only to keep output manageable
fields = list(mapping['kibana_sample_data_ecommerce']['mappings']['properties'].keys())
console.print(f'  Fields in ecommerce mapping: {len(fields)}')
console.print(f'  First 10: {fields[:10]}')

───────────────────────────────────── 2. Index mappings (stored per-snapshot) ─────────────────────────────────────

Fields in ecommerce mapping: 25

First 10: ['category', 'currency', 'customer_birth_date', 'customer_first_name', 'customer_full_name', 
'customer_gender', 'customer_id', 'customer_last_name', 'customer_phone', 'day_of_week']

In [5]:
heading('Index settings (stored per-snapshot)')
settings = es.indices.get_settings(index='kibana_sample_data_ecommerce')
pp(dict(settings['kibana_sample_data_ecommerce']['settings']['index']), 'Index settings')

────────────────────────────────────── Index settings (stored per-snapshot) ───────────────────────────────────────

╭────────────────── Index settings ──────────────────╮
│ {                                                  │
│   "routing": {                                     │
│     "allocation": {                                │
│       "include": {                                 │
│         "_tier_preference": "data_content"         │
│       }                                            │
│     }                                              │
│   },                                               │
│   "number_of_shards": "1",                         │
│   "provided_name": "kibana_sample_data_ecommerce", │
│   "creation_date": "1775497008409",                │
│   "number_of_replicas": "1",                       │
│   "uuid": "u-fNGZztRYOeZWb_ijfVlw",                │
│   "version": {                                     │
│     "created": "9060000"                           │
│   }                                                │
│ }                                                  │
╰────────────────────────────────────────────────────╯

---
## 3. Cluster State (`include_global_state: true`)

When you set `include_global_state: true`, the snapshot also captures the **cluster-wide
configuration layer** — everything above the index level.

In [6]:
heading('3a. Persistent cluster settings')
cluster_settings = es.cluster.get_settings(include_defaults=False)
pp(dict(cluster_settings['persistent']), 'Persistent settings (saved in snapshot)')
info('Transient settings are NOT saved in snapshots.')

───────────────────────────────────────── 3a. Persistent cluster settings ─────────────────────────────────────────

╭─ Persistent settings (saved in snapshot) ─╮
│ {}                                        │
╰───────────────────────────────────────────╯

ℹ Transient settings are NOT saved in snapshots.

In [7]:
# Create some templates so we have something to see
heading('3b. Composable index templates')

es.indices.put_index_template(
    name='training-template',
    body={
        'index_patterns': ['training-*'],
        'template': {
            'settings': {'number_of_shards': 1, 'number_of_replicas': 0},
            'mappings': {'properties': {'@timestamp': {'type': 'date'}, 'message': {'type': 'text'}}},
        },
        'priority': 100,
    },
)
success('Composable template created: training-template')

templates = es.indices.get_index_template(name='training-template')
pp(dict(templates['index_templates'][0]), 'training-template')

───────────────────────────────────────── 3b. Composable index templates ──────────────────────────────────────────

✓ Composable template created: training-template

╭──────────── training-template ────────────╮
│ {                                         │
│   "name": "training-template",            │
│   "index_template": {                     │
│     "index_patterns": [                   │
│       "training-*"                        │
│     ],                                    │
│     "template": {                         │
│       "settings": {                       │
│         "index": {                        │
│           "number_of_shards": "1",        │
│           "number_of_replicas": "0"       │
│         }                                 │
│       },                                  │
│       "mappings": {                       │
│         "properties": {                   │
│           "@timestamp": {                 │
│             "type": "date"                │
│           },                              │
│           "message": {                    │
│             "type": "text"                │
│           }                               │
│         }                                 │
│       }                                   │
│     },                                    │
│     "composed_of": [],                    │
│     "priority": 100,                      │
│     "created_date_millis": 1775497740089, │
│     "modified_date_millis": 1775497740089 │
│   }                                       │
│ }                                         │
╰───────────────────────────────────────────╯

In [8]:
heading('3c. Ingest pipelines')

es.ingest.put_pipeline(
    id='training-pipeline',
    body={
        'description': 'Training demo pipeline',
        'processors': [
            {'set': {'field': 'snapshot_training', 'value': True}},
            {'uppercase': {'field': 'status', 'ignore_missing': True}},
        ],
    },
)
success('Pipeline created: training-pipeline')
pp(dict(es.ingest.get_pipeline(id='training-pipeline')['training-pipeline']), 'training-pipeline')

────────────────────────────────────────────── 3c. Ingest pipelines ───────────────────────────────────────────────

✓ Pipeline created: training-pipeline

╭──────────── training-pipeline ─────────────╮
│ {                                          │
│   "description": "Training demo pipeline", │
│   "processors": [                          │
│     {                                      │
│       "set": {                             │
│         "field": "snapshot_training",      │
│         "value": true                      │
│       }                                    │
│     },                                     │
│     {                                      │
│       "uppercase": {                       │
│         "field": "status",                 │
│         "ignore_missing": true             │
│       }                                    │
│     }                                      │
│   ],                                       │
│   "created_date_millis": 1775498070143,    │
│   "modified_date_millis": 1775498070143    │
│ }                                          │
╰────────────────────────────────────────────╯

In [9]:
heading('3d. ILM policies')

es.ilm.put_lifecycle(
    name='training-ilm-policy',
    body={
        'policy': {
            'phases': {
                'hot':  {'actions': {'rollover': {'max_age': '7d', 'max_size': '5gb'}}},
                'warm': {'min_age': '7d',  'actions': {'shrink': {'number_of_shards': 1}, 'forcemerge': {'max_num_segments': 1}}},
                'cold': {'min_age': '30d', 'actions': {'freeze': {}}},
                'delete': {'min_age': '90d', 'actions': {'delete': {}}},
            },
        },
    },
)
success('ILM policy created: training-ilm-policy')

──────────────────────────────────────────────── 3d. ILM policies ─────────────────────────────────────────────────

/tmp/ipykernel_31901/2994670186.py:3: ElasticsearchWarning: Use of the [max_size] rollover condition found in phase [hot]. This condition has been deprecated in favour of the [max_primary_shard_size] condition and will be removed in a later version
  es.ilm.put_lifecycle(
/tmp/ipykernel_31901/2994670186.py:3: ElasticsearchWarning: The freeze action in ILM is deprecated and will be removed in a future version; this action is already a noop so it can be safely removed. Please remove the freeze action from the 'training-ilm-policy' policy.
  es.ilm.put_lifecycle(


✓ ILM policy created: training-ilm-policy

In [10]:
heading('3e. Stored scripts')

es.put_script(
    id='training-score-script',
    body={
        'script': {
            'lang': 'painless',
            'source': "doc['taxful_total_price'].value * params.multiplier",
        },
    },
)
success('Stored script created: training-score-script')

─────────────────────────────────────────────── 3e. Stored scripts ────────────────────────────────────────────────

✓ Stored script created: training-score-script

In [12]:
# Now create a snapshot with global state and prove all of the above is included
heading('Snapshot with include_global_state: true')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'global-state-snap')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='global-state-snap',
    body={
        'indices': [],
        'include_global_state': True,
        'feature_states': [],          # exclude feature states for now
    },
    wait_for_completion=True,
)

snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='global-state-snap')['snapshots'][0]
pp(snap, 'global-state-snap metadata')

──────────────────────────────────── Snapshot with include_global_state: true ─────────────────────────────────────

╭─────────────────────── global-state-snap metadata ────────────────────────╮
│ {                                                                         │
│   "snapshot": "global-state-snap",                                        │
│   "uuid": "bQ0rWWcyQpuJCricNMQiyA",                                       │
│   "repository": "training-fs-repo",                                       │
│   "version_id": 9060000,                                                  │
│   "version": "9.3.0",                                                     │
│   "indices": [                                                            │
│     ".internal.alerts-dataset.quality.alerts-default-000001",             │
│     ".kibana_search_solution_9.3.0_001",                                  │
│     ".internal.alerts-security.alerts-default-000001",                    │
│     ".ds-ilm-history-7-2026.04.06-000001",                                │
│     ".ml-annotations-000001",                                             │
│     ".apm-custom-link",                                                   │
│     ".apm-source-map",                                                    │
│     ".slo-observability.summary-v3.6",                                    │
│     ".internal.alerts-observability.uptime.alerts-default-000001",        │
│     ".kibana_usage_counters_9.3.0_001",                                   │
│     ".kibana-siem-rule-migrations-prebuiltrules",                         │
│     ".ml-inference-000005",                                               │
│     ".kibana_analytics_9.3.0_001",                                        │
│     ".ds-kibana_sample_data_logs-2026.04.06-000001",                      │
│     ".internal.alerts-stack.alerts-default-000001",                       │
│     ".internal.alerts-ml.anomaly-detection-health.alerts-default-000001", │
│     ".ml-notifications-000002",                                           │
│     ".internal.alerts-security.attack.discovery.alerts-default-000001",   │
│     ".ds-.edr-workflow-insights-default-2026.04.06-000001",               │
│     ".ml-inference-native-000002",                                        │
│     "kibana_sample_data_ecommerce",                                       │
│     ".inference",                                                         │
│     ".apm-agent-configuration",                                           │
│     ".kibana_9.3.0_001",                                                  │
│     ".internal.alerts-ml.anomaly-detection.alerts-default-000001",        │
│     ".security-profile-8",                                                │
│     ".kibana_locks-000001",                                               │
│     ".security-7",                                                        │
│     ".internal.alerts-observability.apm.alerts-default-000001",           │
│     ".internal.alerts-observability.slo.alerts-default-000001",           │
│     ".ds-.kibana-event-log-ds-2026.04.06-000001",                         │
│     ".kibana_alerting_cases_9.3.0_001",                                   │
│     ".secrets-inference",                                                 │
│     "kibana_sample_data_flights",                                         │
│     ".internal.alerts-observability.metrics.alerts-default-000001",       │
│     ".slo-observability.sli-v3.6",                                        │
│     ".ds-.logs-elasticsearch.deprecation-default-2026.04.06-000001",      │
│     ".internal.alerts-transform.health.alerts-default-000001",            │
│     ".kibana_ingest_9.3.0_001",                                           │
│     ".kibana_security_solution_9.3.0_001",                                │
│     ".slo-observability.summary-v3.6.temp",                               │
│     ".kibana_security_session_1",                                         │
│     ".kibana_task_manager_9.3.0_001",                                     │
│     ".integration_kn

---
## 4. Feature States

Feature states allow Elastic Stack features to **hook into the snapshot lifecycle** and save
their own configuration data. Each feature manages its own indices and saved objects.

Feature states are **opt-in** — you must explicitly list them in `feature_states: [...]`
or use `feature_states: ["none"]` to exclude all of them.

In [13]:
heading('4. Available feature states')
features = es.features.get_features()
t = Table('Feature', 'Description')
for f in features['features']:
    t.add_row(f['name'], f.get('description', '—')[:80])
console.print(t)

─────────────────────────────────────────── 4. Available feature states ───────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Feature              ┃ Description                                                          ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ watcher              │ Manages Watch definitions and state                                  │
│ geoip                │ Manages data related to GeoIP database downloader                    │
│ machine_learning     │ Provides anomaly detection and forecasting functionality             │
│ ent_search           │ Manages configuration for Enterprise Search features                 │
│ async_search         │ Manages results of async searches                                    │
│ synonyms             │ Manages synonyms                                                     │
│ kibana               │ Manages Kibana configuration and reports                             │
│ transform            │ Manages configuration and state for transforms                       │
│ logstash_management  │ Enables Logstash Central Management pipeline storage                 │
│ searchable_snapshots │ Manages caches and configuration for searchable snapshots            │
│ security             │ Manages configuration for Security features, such as users and roles │
│ tasks                │ Manages task results                                                 │
│ inference_plugin     │ Inference plugin for managing inference services and inference       │
│ enrich               │ Manages data related to Enrich policies                              │
│ fleet                │ Manages configuration for Fleet                                      │
└──────────────────────┴──────────────────────────────────────────────────────────────────────┘

In [14]:
# Snapshot with ALL feature states
heading('Snapshot with all feature states')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'all-features-snap')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='all-features-snap',
    body={
        'indices': ['kibana_sample_data_*'],
        'include_global_state': True,
        'feature_states': ['kibana', 'security'],
    },
    wait_for_completion=False,
)

snap = wait_for_snapshot(es, FS_REPO_NAME, 'all-features-snap')
console.print(f'  State       : {snap["state"]}')
console.print(f'  Indices     : {snap["indices"]}')
console.print(f'  Feature states saved: {[fs["feature_name"] for fs in snap.get("feature_states", [])]}')

──────────────────────────────────────── Snapshot with all feature states ─────────────────────────────────────────

ℹ Snapshot state: IN_PROGRESS — waiting...

State       : SUCCESS

Indices     : ['.kibana_security_solution_9.3.0_001', '.apm-agent-configuration', 
'.kibana_task_manager_9.3.0_001', 'kibana_sample_data_ecommerce', 'kibana_sample_data_flights', '.security-7', 
'.kibana_9.3.0_001', '.ds-kibana_sample_data_logs-2026.04.06-000001', '.kibana_usage_counters_9.3.0_001', 
'.kibana_search_solution_9.3.0_001', '.apm-custom-link', '.kibana_locks-000001', '.security-profile-8', 
'.kibana_alerting_cases_9.3.0_001', '.kibana_security_session_1', '.kibana_ingest_9.3.0_001', 
'.kibana_analytics_9.3.0_001']

Feature states saved: ['security', 'kibana']

### Feature State: `kibana`

The `kibana` feature state saves all Kibana saved objects — every dashboard, data view,
visualization, alert, connector, space, and more. This is covered in depth in Module 04.

In [15]:
heading('Kibana feature state — what indices are involved?')
# The Kibana feature state saves data from the .kibana* system indices
kibana_indices = es.cat.indices(index='.kibana*', h='index,docs.count,store.size', format='json')
t = Table('Index', 'Docs', 'Size')
for row in kibana_indices:
    t.add_row(row['index'], row['docs.count'], row['store.size'])
console.print(t)

info('All of these .kibana* indices are captured when feature_states includes "kibana".')

──────────────────────────────── Kibana feature state — what indices are involved? ────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┓
┃ Index                                      ┃ Docs ┃ Size    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━┩
│ .ds-.kibana-event-log-ds-2026.04.06-000001 │ 10   │ 43.9kb  │
│ .kibana_ingest_9.3.0_001                   │ 144  │ 478kb   │
│ .kibana_analytics_9.3.0_001                │ 116  │ 1.9mb   │
│ .kibana_security_session_1                 │ 1    │ 6.9kb   │
│ .kibana_usage_counters_9.3.0_001           │ 36   │ 74.2kb  │
│ .kibana_search_solution_9.3.0_001          │ 0    │ 249b    │
│ .kibana_security_solution_9.3.0_001        │ 2    │ 9kb     │
│ .kibana-siem-rule-migrations-integrations  │ 0    │ 249b    │
│ .kibana_task_manager_9.3.0_001             │ 52   │ 160.1kb │
│ .kibana-siem-rule-migrations-prebuiltrules │ 0    │ 249b    │
│ .kibana_9.3.0_001                          │ 34   │ 80.2kb  │
│ .kibana_locks-000001                       │ 0    │ 251b    │
│ .kibana_alerting_cases_9.3.0_001           │ 1    │ 7.5kb   │
└────────────────────────────────────────────┴──────┴─────────┘

ℹ All of these .kibana* indices are captured when feature_states includes "kibana".

### Feature State: `security`

The `security` feature state saves:
- Native users and their password hashes
- Roles and role mappings
- API keys
- Service account tokens
- Security configuration stored in `.security*` indices

In [16]:
heading('Security feature state — system indices')
sec_indices = es.cat.indices(index='.security*', h='index,docs.count,store.size', format='json')
t = Table('Index', 'Docs', 'Size')
for row in sec_indices:
    t.add_row(row['index'], row['docs.count'], row['store.size'])
console.print(t)

───────────────────────────────────── Security feature state — system indices ─────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┓
┃ Index               ┃ Docs ┃ Size    ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━┩
│ .security-7         │ 514  │ 924.9kb │
│ .security-profile-8 │ 1    │ 8.7kb   │
└─────────────────────┴──────┴─────────┘

---
## 5. What Is NOT Saved in a Snapshot

Knowing what's excluded is as important as knowing what's included.

In [17]:
heading('5. What is NOT in a snapshot')

exclusions = [
    ('Transient cluster settings', 'Use persistent settings instead for anything you want to survive a snapshot/restore'),
    ('Registered snapshot repositories', 'You must re-register repos on the target cluster'),
    ('Node configuration files', 'elasticsearch.yml, jvm.options — managed outside ES'),
    ('SSL certificates & keys', 'Manage separately; not stored in ES state'),
    ('Closed indices', 'Closed indices are excluded unless explicitly listed'),
    ('ES installation directory', 'Plugins, binaries — not data'),
    ('Watcher execution history', 'Watcher watches are saved, but execution history indices are not'),
    ('ML results indices (by default)', 'ML jobs are saved; results require explicit index inclusion'),
]

t = Table('Excluded', 'Why / What to do instead')
for item, reason in exclusions:
    t.add_row(item, reason)
console.print(t)

────────────────────────────────────────── 5. What is NOT in a snapshot ───────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Excluded                         ┃ Why / What to do instead                                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Transient cluster settings       │ Use persistent settings instead for anything you want to survive a           │
│                                  │ snapshot/restore                                                             │
│ Registered snapshot repositories │ You must re-register repos on the target cluster                             │
│ Node configuration files         │ elasticsearch.yml, jvm.options — managed outside ES                          │
│ SSL certificates & keys          │ Manage separately; not stored in ES state                                    │
│ Closed indices                   │ Closed indices are excluded unless explicitly listed                         │
│ ES installation directory        │ Plugins, binaries — not data                                                 │
│ Watcher execution history        │ Watcher watches are saved, but execution history indices are not             │
│ ML results indices (by default)  │ ML jobs are saved; results require explicit index inclusion                  │
└──────────────────────────────────┴──────────────────────────────────────────────────────────────────────────────┘

---
## 6. Inspecting Snapshot Metadata in Detail

In [18]:
heading('6. Full snapshot metadata — all-features-snap')

snap = es.snapshot.get(
    repository=FS_REPO_NAME,
    snapshot='all-features-snap',
    verbose=True,
)['snapshots'][0]

# Walk through each field
fields_of_interest = [
    'snapshot', 'uuid', 'version_id', 'version',
    'indices', 'data_streams', 'include_global_state',
    'feature_states', 'state', 'start_time', 'end_time',
    'shards', 'metadata',
]

for field in fields_of_interest:
    val = snap.get(field, '—')
    if isinstance(val, (dict, list)):
        console.print(f'  [bold]{field}[/bold]:')
        console.print(f'    {json.dumps(val, indent=4, default=str)[:300]}')
    else:
        console.print(f'  [bold]{field}[/bold]: {val}')

────────────────────────────────── 6. Full snapshot metadata — all-features-snap ──────────────────────────────────

snapshot: all-features-snap

uuid: gejxTY5nRYWynOdloA9fXw

version_id: 9060000

version: 9.3.0

...

start_time: 2026-04-06T18:00:39.334Z

end_time: 2026-04-06T18:00:39.534Z

shards:

{
    "total": 17,
    "failed": 0,
    "successful": 17
}

metadata: —

## Summary

| Data category | Controlled by | Default |
|--------------|---------------|--------|
| Index data (shards) | `indices` parameter | all open indices |
| Index metadata | always included with index data | — |
| Cluster state | `include_global_state` | `false` |
| Feature states | `feature_states` list | `[]` (none) |

**Next:** [03_creating_snapshots.ipynb](03_creating_snapshots.ipynb)